# 00 — Preparazione del campione

Questo notebook estrae un **campione casuale riproducibile** di `N_SAMPLES` esempi dal dataset
Multi-News pulito ([data/tab/complete.tab](../data/tab/complete.tab), 56.101 righe: le tre split
train/val/test unite, già private delle 115 righe difettose) e lo salva in
`results/sample/sample_{N}_seed{S}.tsv`.

Tutti i notebook dei metodi di summarization (01–04) leggono **lo stesso file campione**, così i
quattro metodi vengono valutati esattamente sugli stessi esempi. Il file conserva la colonna
`split` (train/val/test) di ogni riga: serve per ri-aggregare le metriche per split — in
particolare perché il checkpoint `google/pegasus-multi_news` è stato addestrato sulla split train
di questo stesso dataset (vedi notebook 04).

Il file prodotto è piccolo (~1–2 MB per N=100) e autocontenuto: può essere caricato direttamente
su Google Colab senza portarsi dietro l'intero dataset.

**Eseguire questo notebook una sola volta** (o di nuovo se si cambia `N_SAMPLES`/`SEED`): a parità
di parametri il campione estratto è identico.

In [3]:
# --- Configurazione ---------------------------------------------------------
import sys
from pathlib import Path

N_SAMPLES = 56101   # dimensione del campione (parametrizzabile)
SEED      = 42    # seme per la riproducibilità

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # Su Colab: caricare data/tab/complete.tab in /content (sconsigliato, ~658 MB —
    # questo notebook è pensato per girare in locale nel repository)
    BASE_DIR = Path('/content')
else:
    # In locale: il notebook vive in notebooks/, la root del repo è la cartella sopra
    BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

COMPLETE_TAB = BASE_DIR / 'data' / 'tab' / 'complete.tab'
SAMPLE_DIR   = BASE_DIR / 'results' / 'sample'
SAMPLE_PATH  = SAMPLE_DIR / f'sample_{N_SAMPLES}_seed{SEED}.tsv'

print(f'Sorgente : {COMPLETE_TAB}')
print(f'Campione : {SAMPLE_PATH}')
assert COMPLETE_TAB.exists(), f'File non trovato: {COMPLETE_TAB}'

Sorgente : c:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\data\tab\complete.tab
Campione : c:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\sample\sample_56101_seed42.tsv


## Estrazione del campione

`complete.tab` è in formato Orange: 3 righe di intestazione (nomi, tipi, flag) seguite dalle righe
dati, delimitate da tab e con i campi quotati. Come documentato in
[data/README.md](../data/README.md), i documenti contengono newline e tab letterali che fanno
round-trip **solo** leggendo con il modulo `csv` (mai con uno split ingenuo delle righe).

Il file pesa ~658 MB, quindi viene letto in streaming, in due passate:
1. prima passata: conteggio delle righe dati;
2. estrazione degli indici campionati con `random.sample` (seme fisso) e seconda passata che
   raccoglie solo quelle righe.

In [4]:
import csv
import random
import time

# I documenti più lunghi superano il limite di campo predefinito del modulo csv
csv.field_size_limit(2**31 - 1)

HEADER_LINES = 3  # intestazione Orange: nomi, tipi, flag

def itera_righe_dati(path):
    """Itera le righe dati di un file .tab Orange come (indice_0_based, riga)."""
    with open(path, encoding='utf-8', newline='') as f:
        reader = csv.reader(f, delimiter='\t')
        for _ in range(HEADER_LINES):
            next(reader)
        for i, row in enumerate(reader):
            yield i, row

# Passata 1: conteggio righe dati
t0 = time.time()
n_rows = sum(1 for _ in itera_righe_dati(COMPLETE_TAB))
print(f'Righe dati in complete.tab: {n_rows:,} ({time.time()-t0:.0f} s)')

# Estrazione riproducibile degli indici (ordinati per mantenere l'ordine originale)
random.seed(SEED)
indici = sorted(random.sample(range(n_rows), N_SAMPLES))
indici_set = set(indici)

# Passata 2: raccolta delle righe campionate (colonne: document, summary, split)
t0 = time.time()
campione = []
for i, row in itera_righe_dati(COMPLETE_TAB):
    if i in indici_set:
        document, summary, split = row
        campione.append((i, split, document, summary))
        if len(campione) == N_SAMPLES:
            break
print(f'Righe raccolte: {len(campione)} ({time.time()-t0:.0f} s)')

Righe dati in complete.tab: 56,101 (5 s)
Righe raccolte: 56101 (6 s)


In [5]:
# --- Salvataggio del campione ------------------------------------------------
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
with open(SAMPLE_PATH, 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f, delimiter='\t', quoting=csv.QUOTE_ALL)
    writer.writerow(['row_id', 'split', 'document', 'summary'])
    writer.writerows(campione)

print(f'Campione salvato in {SAMPLE_PATH}')
print(f'Dimensione: {SAMPLE_PATH.stat().st_size / 1e6:.2f} MB')

Campione salvato in c:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\sample\sample_56101_seed42.tsv
Dimensione: 690.37 MB


## Verifiche di integrità

Controlli rapidi sul file appena scritto: numero di righe, distribuzione per split, presenza dei
marcatori attesi nei documenti (newline reali e il separatore di articoli `` ||||| ``) e anteprima
di un esempio.

In [ ]:
from collections import Counter

with open(SAMPLE_PATH, encoding='utf-8', newline='') as f:
    reader = csv.reader(f, delimiter='\t')
    header = next(reader)
    righe = list(reader)

assert header == ['row_id', 'split', 'document', 'summary']
assert len(righe) == N_SAMPLES, f'Attese {N_SAMPLES} righe, trovate {len(righe)}'

per_split = Counter(r[1] for r in righe)
con_newline = sum(1 for r in righe if '\n' in r[2])
con_separatore = sum(1 for r in righe if '|||||' in r[2])

print(f'Righe: {len(righe)}')
print(f'Distribuzione per split: {dict(per_split)}')
print(f'Documenti con newline reali: {con_newline}/{N_SAMPLES}')
print(f'Documenti multi-articolo (contengono \'|||||\'): {con_separatore}/{N_SAMPLES}')

esempio = righe[0]
print(f'\n--- Esempio (row_id={esempio[0]}, split={esempio[1]}) ---')
print(f'Documento (primi 300 caratteri):\n{esempio[2][:300]}...')
print(f'\nRiassunto di riferimento (primi 300 caratteri):\n{esempio[3][:300]}...')

Righe: 56101
Distribuzione per split: {'train': 44880, 'val': 5611, 'test': 5610}
Documenti con newline reali: 55974/56101
Documenti multi-articolo (contengono '|||||'): 55520/56101

--- Esempio (row_id=0, split=train) ---
Documento (primi 300 caratteri):
National Archives 
 
 Yes, it’s that time again, folks. It’s the first Friday of the month, when for one ever-so-brief moment the interests of Wall Street, Washington and Main Street are all aligned on one thing: Jobs. 
 
 A fresh update on the U.S. employment situation for January hits the wires at...

Riassunto di riferimento (primi 300 caratteri):
– The unemployment rate dropped to 8.2% last month, but the economy only added 120,000 jobs, when 203,000 new jobs had been predicted, according to today's jobs report. Reaction on the Wall Street Journal's MarketBeat Blog was swift: "Woah!!! Bad number." The unemployment rate, however, is better ne...


: 